# CIFAR-10 STAQ

A notebook for the CIFAR-10 workflow:

- load config, CLIP, concepts, and data
- load or train Concept-QA
- load or train baseline VIP and STAQ
- replay tiny-start cases
- mine confidence-stop contrasts

In [ ]:
%pip install -e ..

In [ ]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from staq.analysis import mine_confidence_stop_contrasts, plot_rollout_comparisons, sample_intuition_replays
from staq.config import Cifar10StaqConfig, default_paths
from staq.core import (
    build_concept_dictionary,
    concept_answers_batch,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_concepts,
    load_vip_bundle,
    make_sensitive_mask,
    save_bundle_checkpoint,
)
from staq.data import get_cifar10_datasets, get_cifar10_loaders, get_raw_cifar10_dataset
from staq.models import ConceptNet2
from staq.sensitive_labels import build_sensitive_index, build_sensitive_labels
from staq.training import HistorySamplingConfig, build_staq_models, fit_concept_qa, fit_staq, seed_everything
from staq.training.concept_qa import load_gpt_answers

In [ ]:
repo_root = Path.cwd().resolve()
if not (repo_root / "staq").exists() and (repo_root.parent / "staq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()

config = Cifar10StaqConfig()
device = config.device
seed_everything(config.random_seed)

print(f"repo_root: {repo_root}")
print(f"artifacts_root: {paths.artifacts_root}")
print(f"device: {device}")

In [ ]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
concepts = load_concepts(paths.concept_file)
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sens_idx = build_sensitive_index(concepts)
sensitive_mask = make_sensitive_mask(config.max_queries, sens_idx, device)

train_ds, test_ds = get_cifar10_datasets(transform=preprocess, root=paths.data_root)
train_loader, test_loader = get_cifar10_loaders(
    transform=preprocess,
    root=paths.data_root,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
)
raw_test_ds = get_raw_cifar10_dataset(paths.data_root, train=False)

print(f"# concepts: {len(concepts)}")
print(f"# sensitive concepts: {len(sens_idx)}")
print([concepts[i] for i in sens_idx.tolist()])

In [ ]:
qa_checkpoint = paths.checkpoints_root / "concept_qa_cifar10.pt"
qa_source = qa_checkpoint
if qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(qa_checkpoint, device=device)
elif paths.bootstrap_concept_qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(paths.bootstrap_concept_qa_checkpoint, device=device)
    qa_source = paths.bootstrap_concept_qa_checkpoint
elif paths.gpt_answers_file.exists():
    gpt_answers = load_gpt_answers(paths.gpt_answers_file)
    qa_model = ConceptNet2().to(device)
    qa_optimizer = torch.optim.SGD(qa_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    qa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(qa_optimizer, T_max=200)
    qa_history = fit_concept_qa(
        model=qa_model,
        train_loader=train_loader,
        eval_loader=test_loader,
        optimizer=qa_optimizer,
        scheduler=qa_scheduler,
        num_epochs=2,
        model_clip=model_clip,
        dictionary=dictionary,
        gpt_answers=gpt_answers,
        clip_device=device,
        train_device=device,
    )
    torch.save(qa_model.state_dict(), qa_checkpoint)
    with open(paths.runs_root / "concept_qa_cifar10_history.json", "w", encoding="utf-8") as handle:
        json.dump(qa_history, handle, indent=2)
    answering_model = qa_model.eval()
else:
    raise FileNotFoundError("No local Concept-QA checkpoint, bootstrap checkpoint, or GPT answers file available.")

print(f"Concept-QA ready from: {qa_source}")

In [ ]:
def load_or_train_run(
    run_name,
    lambda_adv,
    alpha_sens,
    min_history=config.min_history,
    max_history=config.max_history,
    non_sensitive_only=config.non_sensitive_history_only,
    epochs=2,
):
    ckpt_path = paths.checkpoints_root / f"{run_name}_best.pt"
    if ckpt_path.exists():
        return load_vip_bundle(ckpt_path, device=device, max_queries=config.max_queries, num_classes=config.num_classes)

    actor_checkpoint = str(paths.bootstrap_actor_checkpoint) if paths.bootstrap_actor_checkpoint.exists() else None
    classifier_checkpoint = str(paths.bootstrap_classifier_checkpoint) if paths.bootstrap_classifier_checkpoint.exists() else None

    actor, classifier, s_head = build_staq_models(
        max_queries=config.max_queries,
        num_classes=config.num_classes,
        device=device,
        actor_eps=config.actor_eps,
        actor_checkpoint=actor_checkpoint,
        classifier_checkpoint=classifier_checkpoint,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=config.learning_rate,
    )
    history_config = HistorySamplingConfig(
        min_history=min_history,
        max_history=max_history,
        non_sensitive_only=non_sensitive_only,
    )
    history, best = fit_staq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=train_loader,
        test_loader=test_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=history_config,
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_adv=lambda_adv,
        alpha_sens=alpha_sens,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=epochs,
        max_train_batches=60 if device.type == "cpu" else None,
        max_test_batches=30 if device.type == "cpu" else None,
    )
    save_bundle_checkpoint(
        checkpoint_path=ckpt_path,
        metadata={
            "run_name": run_name,
            "lambda_adv": lambda_adv,
            "alpha_sens": alpha_sens,
            "best_test_acc": best["test_acc"],
            "best_epoch": best["epoch"],
            "history_config": {
                "min_history": history_config.min_history,
                "max_history": history_config.max_history,
                "non_sensitive_only": history_config.non_sensitive_only,
            },
            "actor_state_dict": best["actor_state_dict"],
            "classifier_state_dict": best["classifier_state_dict"],
            "s_head_state_dict": best["s_head_state_dict"],
        },
    )
    with open(paths.runs_root / f"{run_name}_history.json", "w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    return load_vip_bundle(ckpt_path, device=device, max_queries=config.max_queries, num_classes=config.num_classes)


baseline_bundle = load_or_train_run("baseline_vip", lambda_adv=0.0, alpha_sens=0.0)
staq_bundle = load_or_train_run("lam_0.80", lambda_adv=0.8, alpha_sens=0.0, epochs=20)

print(baseline_bundle["ckpt_path"])
print(staq_bundle["ckpt_path"])

def answer_builder(images):
    return concept_answers_batch(
        images=images,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        clip_device=device,
        train_device=device,
        threshold=config.threshold_for_binarization,
    )

In [ ]:
intuition_records = sample_intuition_replays(
    dataset=test_ds,
    answer_builder=answer_builder,
    baseline_bundle=baseline_bundle,
    staq_bundle=staq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=raw_test_ds.classes,
    num_cases=6,
    pool_size=400 if device.type == "cpu" else 1500,
    num_trials=2,
    min_history=1,
    max_history=2,
)

intuition_fig = plot_rollout_comparisons(
    records=intuition_records,
    raw_dataset=raw_test_ds,
    output_path=paths.figures_root / "cifar10_intuition_replay_examples.png",
    title_prefix="tiny-start replay",
)

print(f"Saved tiny-start replay figure: {intuition_fig}")

In [ ]:
contrast_result = mine_confidence_stop_contrasts(
    loader=test_loader,
    answer_builder=answer_builder,
    baseline_bundle=baseline_bundle,
    staq_bundle=staq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=raw_test_ds.classes,
    threshold=config.confidence_threshold,
    max_steps=config.confidence_max_steps,
    min_history=1,
    max_history=2,
    max_search_samples=400 if device.type == "cpu" else None,
    num_trials=4,
)

plot_records = contrast_result["plot_candidates"] if contrast_result["plot_candidates"] else contrast_result["selected_candidates"]
plot_warning = None
if not contrast_result["plot_candidates"] and contrast_result["selected_candidates"]:
    plot_warning = "Showing weak fallback examples only."

contrast_fig = None
if plot_records:
    contrast_fig = plot_rollout_comparisons(
        records=plot_records,
        raw_dataset=raw_test_ds,
        output_path=paths.figures_root / "cifar10_confidence_stop_contrasts.png",
        title_prefix="confidence-stop contrast",
        bucket_name=contrast_result["plot_bucket"] if contrast_result["plot_candidates"] else contrast_result["selected_bucket"],
        warning=plot_warning,
    )

with open(paths.runs_root / "cifar10_confidence_stop_contrasts.json", "w", encoding="utf-8") as handle:
    json.dump(
        {
            "stats": contrast_result["stats"],
            "bucket_counts": contrast_result["bucket_counts"],
            "selected_bucket": contrast_result["selected_bucket"],
            "plot_bucket": contrast_result["plot_bucket"],
            "selected_candidates": contrast_result["selected_candidates"],
        },
        handle,
        indent=2,
    )

print("bucket_counts:", contrast_result["bucket_counts"])
print("stats:", contrast_result["stats"])
if contrast_fig is None:
    print("No contrast figure was created because the miner found no plotable candidates.")
else:
    print(f"Saved contrast figure: {contrast_fig}")